In [ ]:
import os
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "reproduce.py").is_file())
os.chdir(ROOT)
import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import matplotlib.gridspec as gridspec
PROJECT_PATH = Path("data/genomics") 

In [ ]:
meta = pd.read_csv(PROJECT_PATH / "reference/metadata_complete.csv")
meta['population'] = meta['Strain'] + '_' + meta['Culture'].astype(str).str.zfill(2)
meta

In [ ]:
dfs = []

for i, row in meta.iterrows():
    df = pd.read_csv(PROJECT_PATH / f"out/{row['FolderDate']}/{row['source_file']}/output/output.gd.tsv", sep='\t')
    df["Strain"] = row['Strain']
    df["Culture"] = row['Culture']
    df["Day"] = int(row['Day'])
    dfs.append(df)

df = pd.concat(dfs)
df

In [ ]:
def get_lineage_info(meta, select_lineage):
    """
    Retrieves populations and timepoints for a specific lineage.
    
    Parameters:
        meta (DataFrame): Metadata containing lineage information.
        select_lineage (str): The lineage to select.
    
    Returns:
        tuple: A tuple containing:
            - pops (ndarray): Array of populations for the selected lineage.
            - timepoints (list): List of timepoints for the selected lineage.
    """
    pops = meta.query(f'Strain=="{select_lineage}"')['population'].unique()
    timepoints = sorted(meta.query(f'Strain=="{select_lineage}"')['Day'].unique())
    return pops, timepoints



In [ ]:
def traceAlleleFreq(pop, meta, project_path, min_freq=0.33, verbose=True):
    """
    Traces allele frequencies for a given population, filtering out background mutations.

    Parameters:
        pop (str): Population to process.
        meta (DataFrame): Metadata containing population details.
        project_path (Path): Base path for the project files.
        min_freq (float): Minimum frequency threshold for tracking mutations.

    Returns:
        DataFrame: Traced allele frequency data with mutation details.
    """
    D = []
    sorted_meta = meta.query(f'population=="{pop}"').sort_values(by='Day')

    for i, row in sorted_meta.iterrows():
        df = pd.read_csv(project_path / f"out/{row['FolderDate']}/{row['source_file']}/output/output.gd.tsv", sep='\t')
        D.append(df)

    # Remove mutations detected in the wild-type background
    wt_background = D[0].loc[D[0].frequency > 0.01, 'position']
    D_ = [df[~df['position'].isin(wt_background)] for df in D]

    tracked_positions = set(
        pos for df in D_ 
        for pos in df.loc[df['frequency'] > min_freq, 'position']
    )
    if verbose:
        print(f"Population {pop} # of tracked mutations: {len(tracked_positions)}")
    

    # Trace the frequency of those mutations
    T = []
    for pos in tracked_positions:
        freq = []
        for i in range(len(D_)):
            d = D_[i]
            ind = d['position'] == pos

            if sum(ind) < 1:
                freq.append(0)
            else:
                freq.append(d.loc[ind, 'frequency'].values[0])
                gene_name = d.loc[ind, 'gene_name'].values[0]
                gene_product = d.loc[ind, 'gene_product'].values[0]
                aa_ref_seq = d.loc[ind, 'aa_ref_seq'].values[0]
                aa_new_seq = d.loc[ind, 'aa_new_seq'].values[0]
                new_seq = d.loc[ind, 'new_seq'].values[0]
                codon_ref_seq = d.loc[ind, 'codon_ref_seq'].values[0]
                aa_pos = d.loc[ind, 'aa_position'].values[0]
                gene_pos = d.loc[ind, 'gene_position'].values[0]
                mut_cat = d.loc[ind, 'mutation_category'].values[0]

        T.append({
            'position': pos, 
            'freq': np.round(freq, 2), 
            'gene_name': gene_name, 
            'gene_product': gene_product,
            'aa_ref_seq': aa_ref_seq, 
            'aa_new_seq': aa_new_seq,
            'aa_pos': aa_pos, 
            'gene_pos': gene_pos, 
            'mut_cat': mut_cat,
            'new_seq': new_seq, 
            'codon_ref_seq': codon_ref_seq,
            'population': pop
        })

    T = pd.DataFrame(T)
    T.fillna({'aa_ref_seq': '', 'aa_new_seq': '', 'aa_pos': ''}, inplace=True)

    nsi = T['aa_pos'] == ''
    T.loc[nsi, 'label'] = T.loc[nsi, 'gene_name'] + ' ' + T.loc[nsi, 'mut_cat']
    T.loc[~nsi, 'label'] = (
        T.loc[~nsi, 'gene_name'] + ' ' +
        T.loc[~nsi, 'aa_ref_seq'] +
        T.loc[~nsi, 'aa_pos'].astype(str).str.replace(r'\.0', '', regex=True) +
        T.loc[~nsi, 'aa_new_seq']
    )

    T.sort_values(by=['mut_cat', 'gene_name'], ascending=False, inplace=True)
    T.reset_index(inplace=True, drop=True)
    
    return T


In [ ]:
def collect_allele_frequencies(pops, meta, project_path, min_freq=0.1):
    """
    Collects allele frequency data for a list of populations.
    
    Parameters:
        pops (list): List of populations to process.
        meta (DataFrame): Metadata containing population details.
        project_path (Path): Base path for the project files.
        min_freq (float): Minimum frequency threshold for tracking mutations.
    
    Returns:
        list: A list of DataFrames containing allele frequency data for each population.
    """
    allele_frequencies = []
    for pop in pops:
        allele_data = traceAlleleFreq(pop, meta, project_path, min_freq=min_freq)
        allele_frequencies.append(allele_data)
    return allele_frequencies


In [ ]:
def combine_allele_frequencies(allele_frequencies):
    """
    Combines allele frequency data from multiple populations into a single DataFrame.
    
    Parameters:
        allele_frequencies (list): List of DataFrames containing allele frequency data for each population.
    
    Returns:
        DataFrame: Combined DataFrame with an added 'Pop' column and computed 'last_freq' values.
    """
    combined = []
    for ix, df in enumerate(allele_frequencies):
        dff = df.copy()
        # dff['Pop'] = ix + 1  # Add population index
        dff['Pop'] = dff['population']
        combined.append(dff)
    
    combined_df = pd.concat(combined)
    combined_df['last_freq'] = combined_df['freq'].apply(lambda x: x[-1])  # Add last frequency
    return combined_df





In [ ]:
def plot_combined_heatmap(combined_df, output_dir, file_name="combined_lineages_heatmap.png", save_plot=True):
    """
    Plots a heatmap of mutation frequencies with axes switched: mutations on the y-axis
    and populations on the x-axis.
    """
    import matplotlib.colors as mcolors
    import matplotlib.gridspec as gridspec
    import re
    plt.rcParams["font.family"] = "Nimbus Roman"
    alt_labels = {
        'glvC small_indel': r'$\it{glvC}$ indel',
        'fimB/fimE mobile_element_insertion': r'$\it{fimB/fimE}$ insertion',
        'rpoZ small_indel': r'$\it{rpoZ}$ indel',
        'selB small_indel': r'$\it{selB}$ indel',
        'acrR mobile_element_insertion': r'$\it{acrR}$ insertion',
        'gatA mobile_element_insertion': r'$\it{gatA}$ insertion',
        'ftsH small_indel': r'$\it{ftsH}$ indel',
        'ompC mobile_element_insertion': r'$\it{ompC}$ insertion',
        'rpoB small_indel': r'$\it{rpoB}$ indel',
        'pssL small_indel': r'$\it{pssL}$ indel',
        'mutL small_indel': r'$\it{mutL}$ indel',
        'ppiC/rep snp_intergenic': r'$\it{ppiC/rep}$ snp intergenic',
        'glnL/glnA snp_intergenic': r'$\it{glnL/glnA}$ snp intergenic',
        'iscR/trmJ snp_intergenic': r'$\it{iscR/trmJ}$ snp intergenic',
        'appY/ompT small_indel': r'$\it{appY/ompT}$ indel',
        'ompC small_indel': r'$\it{ompC}$ indel',
        'tyrR small_indel': r'$\it{tyrR}$ indel',
        'tas small_indel': r'$\it{tas}$ indel',
        'phoQ small_indel': r'$\it{phoQ}$ indel',
        'mgtL/mgtA small_indel': r'$\it{mgtL/mgtA}$ indel',
        'arcA small_indel': r'$\it{arcA}$ indel',
        'emrD small_indel': r'$\it{emrD}$ indel',
        'trkH small_indel': r'$\it{trkH}$ indel',
        'marR small_indel': r'$\it{marR}$ indel',
        'marC/marR small_indel': r'$\it{marC/marR}$ indel',
        'accB/accC snp_intergenic': r'$\it{accB/accC}$ snp intergenic',
        'yiaF/yiaG snp_intergenic': r'$\it{yiaF/yiaG}$ snp intergenic',
        'ygiZ/mdaB snp_intergenic': r'$\it{ygiZ/mdaB}$ snp intergenic',
        'ftp/ompC snp_intergenic': r'$\it{ftp/ompC}$ snp intergenic',
        'hns/tdk mobile_element_insertion': r'$\it{hns/tdk}$ insertion',
        'yfbN/yfbO mobile_element_insertion': r'$\it{yfbN/yfbO}$ insertion',
        'insH1/mmuP snp_intergenic': r'$\it{insH1/mmuP}$ snp intergenic',
        'insH5/lomR snp_intergenic': r'$\it{insH5/lomR}$ snp intergenic',
        'phoQ mobile_element_insertion': r'$\it{phoQ}$ insertion',
        'yhaC/rnpB snp_intergenic': r'$\it{yhaC/rnpB}$ snp intergenic',
        'lysO/aqpZ snp_intergenic': r'$\it{lysO/aqpZ}$ snp intergenic',
        'gltP/yjcO snp_intergenic': r'$\it{gltP/yjcO}$ snp intergenic',
        'csgD/csgB snp_intergenic': r'$\it{csgD/csgB}$ snp intergenic',
        'yhfG/ppiA snp_intergenic': r'$\it{yhfG/ppiA}$ snp intergenic',
        'yfjW/yfjX snp_intergenic': r'$\it{yfjW/yfjX}$ snp intergenic',
        'ybiO/glnQ snp_intergenic': r'$\it{ybiO/glnQ}$ snp intergenic',
        'ypfG/nudK snp_intergenic': r'$\it{ypfG/nudK}$ snp intergenic',
        'valW/ydhR snp_intergenic': r'$\it{valW/ydhR}$ snp intergenic',
        'yiiG/frvR snp_intergenic': r'$\it{yiiG/frvR}$ snp intergenic',
        'insB9–[crl] large_deletion': r'$\it{insB9–[crl]}$ deletion',
        'ftsI|ftsO L|NA376|NAS|NA': r'$\it{ftsI|ftsO}$ snp 376',
        'yibA/yibG snp_intergenic': r'$\it{yibA/yibG}$ snp intergenic',
        'ftsI|ftsO D|NA409|NAG|NA': r'$\it{ftsI/ftsO}$ snp 409',
        'cheA|cheA D|D324|421G|G': r'$\it{cheA}$ snp 324/421',
        'cheA|cheA T|T400|497A|A': r'$\it{cheA}$ snp 400/497',
        'mrcB|mrcB T|T702|657S|S': r'$\it{mrcB}$ snp',
        'yffN/yffO snp_intergenic': r'$\it{yffN/yffO}$ snp intergenic',
        'insG/nanX snp_intergenic': r'$\it{insG/nanX}$ snp intergenic',
        'livK/panZ snp_intergenic': r'$\it{livK/panZ}$ snp intergenic',
        'yebK mobile_element_insertion': r'$\it{yebK}$ insertion',
        'yicC mobile_element_insertion': r'$\it{yicC}$ insertion',
        'mgrR snp_noncoding': r'$\it{mgrR}$ snp noncoding',
        'aroK small_indel': r'$\it{aroK}$ indel',
        'digH small_indel': r'$\it{digH}$ indel',
        '[metR]–[ysgA] large_deletion': r'$\it{metR/ysgA}$ large_deletion',
        'aaeB T362A': 'AaeB T362A',
        'aaeR V79I': 'AaeR V79I',
        'abgT L394F': 'AbgT L394F',
        'abgT Y128Y': 'AbgT Y128Y',
        'aceK V471G': 'AceK V471G',
        'acrF G690G': 'AcrF G690G',
        'adhE G194G': 'AdhE G194G',
        'adiY/adiA snp_intergenic': '$\\it{adiY/adiA}$ snp intergenic',
        'alaS G196G': 'AlaS G196G',
        'alsK C305C': 'AlsK C305C',
        'ampH/sbmA mobile_element_insertion': '$\\it{ampH/sbmA}$ IS insertion',
        'apaH small_indel': '$\\it{apaH}$ indel',
        'arcA L65F': 'Arca L65F',
        'arcB G258C': 'ArcB G258C',
        'aroG V171A': 'AroG V171A',
        'arpA D661A': 'ArpA D661A',
        'artQ/artI snp_intergenic': '$\\it{artQ/artI}$ snp intergenic',
        'asmA S185*': 'AsmA S185*',
        'asnC Y82H': 'AsnC Y82H',
        'atoC S366P': 'AtoC S366P',
        'baeS F3L': 'BaeS F3L',
        'bamD P32Q': 'BamD P32Q',
        'bcsA R521Q': 'BcsA R521Q',
        'bioH S225C': 'BioH S225C',
        'bluF I39L': 'BluF I39L',
        'carB E103E': 'CarB E103E',
        'clsB G277G': 'ClsB G277G',
        'clsB P261S': 'ClsB P261S',
        'cmk P13S': 'Cmk P13S',
        'coaD F29S': 'CoaD F29S',
        'coaD V67G': 'CoaD V67G',
        'corA H75N': 'CorA H75N',
        'corA L50Q': 'CorA L50Q',
        'cpxA I12L': 'CpxA I12L',
        'cpxA R106L': 'CpxA R106L',
        'cpxA T279A': 'CpxA T279A',
        'cpxA small_indel': '$\\it{cpxA}$ indel',
        'cra S18N': 'Cra S18N',
        'crp F15S': 'Crp F15S',
        'crp I52F': 'Crp I52F',
        'cstA A392S': 'CstA A392S',
        'cusA S587S': 'CusA S587S',
        'cyaA large_deletion': '$\\it{cyaA}$ large deletion',
        'cysN R100H': 'CysN R100H',
        'cysQ T175A': 'CysQ T175A',
        'cysS Y298S': 'CysS Y298S',
        'dcp mobile_element_insertion': '$\\it{dcp}$ IS insertion',
        'dhaR V139A': 'DhaR V139A',
        'digH W125*': 'DigH W125*',
        'dnaQ I97I': 'DnaQ I97I',
        'eco/mqo snp_intergenic': '$\\it{eco/mqo}$ snp intergenic',
        'elaD C252C': 'ElaD C252C',
        'epmA N129S': 'EpmA N129S',
        'eutH P393Q': 'EutH P393Q',
        'eutQ V181A': 'EutQ V181A',
        'fabF/pabC snp_intergenic': '$\\it{fabF/pabC}$ snp intergenic',
        'fadB K305R': 'FadB K305R',
        'fdrA I220T': 'FdrA I220T',
        'feaB D41E': 'FeaB D41E',
        'feoB S549P': 'FeoB S549P',
        'fimE mobile_element_insertion': '$\\it{fimE}$ IS insertion',
        'fimE/fimA small_indel': '$\\it{fimE/fimA}$ indel',
        'fimH T247M': 'FimH T247M',
        'fixA A106A': 'FixA A106A',
        'flu/yeeR snp_intergenic': '$\\it{flu/yeeR}$ snp intergenic',
        'ftsW I226V': 'FtsW I226V',
        'fusA A608V': 'FusA A608V',
        'fusA A678V': 'FusA A678V',
        'fusA F593L': 'FusA F593L',
        'fusA F605L': 'FusA F605L',
        'fusA G117C': 'FusA G117C',
        'fusA G46C': 'FusA G46C',
        'fusA G556V': 'FusA G556V',
        'fusA G60C': 'FusA G60C',
        'fusA I61M': 'FusA I61M',
        'fusA P610L': 'FusA P610L',
        'fusA P610T': 'FusA P610T',
        'fusA R59H': 'FusA R59H',
        'fusA T668A': 'FusA T668A',
        'fusA V116F': 'FusA V116F',
        'gadF/mdtE snp_intergenic': '$\\it{gadF/mdtE}$ snp intergenic',
        'gadX mobile_element_insertion': '$\\it{gadX}$ IS insertion',
        'galS D68G': 'GalS D68G',
        'gatD/gatB mobile_element_insertion': '$\\it{gatD/gatB}$ IS insertion',
        'glcB S8S': 'GlcB S8S',
        'glnE A753T': 'GlnE A753T',
        'glnH A17V': 'GlnH A17V',
        'glpE/glpD snp_intergenic': '$\\it{glpE/glpD}$ snp intergenic',
        'glpF A201T': 'GlpF A201T',
        'glpF G176A': 'GlpF G176A',
        'glpF G176S': 'GlpF G176S',
        'glpF I187V': 'GlpF I187V',
        'glpF V52E': 'GlpF V52E',
        'glpF Y138S': 'GlpF Y138S',
        'glpK mobile_element_insertion': '$\\it{glpK}$ IS insertion',
        'glpK small_indel': '$\\it{glpK}$ indel',
        'glpT D88E': 'GlpT D88E',
        'gsiA A299P': 'GsiA A299P',
        'gsk F3I': 'Gsk F3I',
        'gspD D55A': 'GspD D55A',
        'gspG mobile_element_insertion': '$\\it{gspG}$ IS insertion',
        'gyrA A119E': 'GyrA A119E',
        'gyrA D628G': 'GyrA D628G',
        'gyrA D87Y': 'GyrA D87Y',
        'gyrA G81D': 'GyrA G81D',
        'gyrA S83L': 'GyrA S83L',
        'gyrA S83W': 'GyrA S83W',
        'hipA P86L': 'HipA P86L',
        'hipA S97G': 'HipA S97G',
        'hldE small_indel': '$\\it{hldE}$ indel',
        'hlyE E42D': 'HlyE E42D',
        'hpt V58V': 'Hpt V58V',
        'hscA L103P': 'HscA L103P',
        'hyaE A94V': 'HyaE A94V',
        'hycG S75S': 'HycG S75S',
        'hypE N3N': 'HypE N3N',
        'icd A386A': 'Icd A386A',
        'icd H366H': 'Icd H366H',
        'icd K387K': 'Icd K387K',
        'icd L375M': 'Icd L375M',
        'icd N385N': 'Icd N385N',
        'icd T370T': 'Icd T370T',
        'igaA L652P': 'IgaA L652P',
        'ilvL/ilvX snp_intergenic': '$\\it{ilvL/ilvX}$ snp intergenic',
        'ilvY V201A': 'IlvY V201A',
        'kduI Y137H': 'KduI Y137H',
        'lapC H20R': 'LapC H20R',
        'leuB I134T': 'LeuB I134T',
        'leuX snp_noncoding': '$\\it{leuX}$ snp noncoding',
        'lon L83L': 'Lon L83L',
        'lpxC D117D': 'LpxC D117D',
        'manA mobile_element_insertion': '$\\it{manA}$ IS insertion',
        'manZ/yobD snp_intergenic': '$\\it{manZ/yobD}$ snp intergenic',
        'marR Q117*': 'MarR Q117*',
        'mcrB/symE snp_intergenic': '$\\it{mcrB/symE}$ snp intergenic',
        'mdfA M173V': 'MdfA M173V',
        'mdtA V227M': 'MdtA V227M',
        'mepM G136D': 'MepM G136D',
        'metC V151A': 'MetC V151A',
        'metG M652K': 'MetG M652K',
        'metL S631T': 'MetL S631T',
        'mgtT Y23S': 'MgtT Y23S',
        'mltB C19*': 'MltB C19*',
        'mltB E195K': 'MltB E195K',
        'mltB F121I': 'MltB F121I',
        'moaE G36G': 'MoaE G36G',
        'mqo/yojI snp_intergenic': '$\\it{mqo/yojI}$ snp intergenic',
        'mraY Y21C': 'MraY Y21C',
        'mrdA A329S': 'MrdA A329S',
        'mscK F931C': 'MscK F931C',
        'mscM G160V': 'MscM G160V',
        'msrQ W167R': 'MsrQ W167R',
        'nadK T200P': 'NadK T200P',
        'narH L111P': 'NarH L111P',
        'narU/yddK small_indel': '$\\it{narU/yddK}$ indel',
        'narW E188G': 'NarW E188G',
        'nikA Y322C': 'NikA Y322C',
        'obgE Q71K': 'ObgE Q71K',
        'opgH S614P': 'OpgH S614P',
        'osmY L198Q': 'OsmY L198Q',
        'paaF Y102C': 'PaaF Y102C',
        'paaG S241C': 'PaaG S241C',
        'paaH G313G': 'PaaH G313G',
        'paaH G437S': 'PaaH G437S',
        'paaK P128S': 'PaaK P128S',
        'pdeI L619L': 'PdeI L619L',
        'pgaC E86G': 'PgaC E86G',
        'pgaC G113G': 'PgaC G113G',
        'pgaC I81L': 'PgaC I81L',
        'pgaC I94I': 'PgaC I94I',
        'pgsA V44E': 'PgsA V44E',
        'pheS A14P': 'PheS A14P',
        'phnK T242T': 'PhnK T242T',
        'phoP R67H': 'PhoP R67H',
        'phoQ G39C': 'PhoQ G39C',
        'phoQ IS2 insertion': '$\\it{phoQ}$ IS2 insertion',
        'phoQ IS5 insertion': '$\\it{phoQ}$ IS5 insertion',
        'pitA T184T': 'PitA T184T',
        'plsB A194T': 'PlsB A194T',
        'pncB T340A': 'PncB T340A',
        'ppa/ytfQ snp_intergenic': '$\\it{ppa/ytfQ}$ snp intergenic',
        'ppiA S178*': 'PpiA S178*',
        'ppk S250S': 'Ppk S250S',
        'prlC S491S': 'PrlC S491S',
        'proA G409G': 'ProA G409G',
        'prs A114V': 'Prs A114V',
        'prs A11S': 'Prs A11S',
        'prs A76V': 'Prs A76V',
        'prs I151T': 'Prs I151T',
        'prs R79C': 'Prs R79C',
        'ptsG T335R': 'PtsG T335R',
        'ptsG mobile_element_insertion': '$\\it{ptsG}$ mobile_element_insertion',
        'ptsG small_indel': '$\\it{ptsG}$ small_indel',
        'ptsH L47P': 'PtsH L47P',
        'ptsI L199P': 'PtsI L199P',
        'punC H327R': 'PunC H327R',
        'purM Q340Q': 'PurM Q340Q',
        'purT F300L': 'PurT F300L',
        'puuA D431D': 'PuuA D431D',
        'rapA W549*': 'RapA W549*',
        'rbsA S157R': 'RbsA S157R',
        'recF I355T': 'RecF I355T',
        'relB P45S': 'RelB P45S',
        'rfe T235A': 'Rfe T235A',
        'rhsC D745G': 'RhsC D745G',
        'rihB mobile_element_insertion': '$\\it{rihB}$ IS insertion',
        'rlmB D59E': 'RlmB D59E',
        'rlmL R138C': 'RlmL R138C',
        'rnb R279H': 'Rnb R279H',
        'rpe L138R': 'Rpe L138R',
        'rplJ P89S': 'RplJ P89S',
        'rpoB D1064G': 'RpoB D1064G',
        'rpoB D1064N': 'RpoB D1064N',
        'rpoB E562D': 'RpoB E562D',
        'rpoB E562K': 'RpoB E562K',
        'rpoB H1237L': 'RpoB H1237L',
        'rpoB I870T': 'RpoB I870T',
        'rpoB M1290R': 'RpoB M1290R',
        'rpoB T563A': 'RpoB T563A',
        'rpoC D348A': 'RpoC D348A',
        'rpoC D830G': 'RpoC D830G',
        'rpoD A447P': 'RpoD A447P',
        'rpoD D96A': 'RpoD D96A',
        'rpoD V454A': 'RpoD V454A',
        'rssB/galU snp_intergenic': '$\\it{rssB/galU}$ snp intergenic',
        'sapF R158S': 'SapF R158S',
        'sbmA Q189*': 'SbmA Q189*',
        'sbmA V163E': 'SbmA V163E',
        'sbmA small_indel': '$\\it{sbmA}$ indel',
        'sbp L222L': 'Sbp L222L',
        'sdaA G378G': 'SdaA G378G',
        'selA N204S': 'SelA N204S',
        'serC L14L': 'SerC L14L',
        'serW mobile_element_insertion': '$\\it{serW}$ IS insertion',
        'sgbH A9T': 'SgbH A9T',
        'sgrR G118R': 'SgrR G118R',
        'sodC G103G': 'SodC G103G',
        'speE small_indel': '$\\it{speE}$ indel',
        'spoT small_indel': '$\\it{spoT}$ indel',
        'ssrA snp_noncoding': '$\\it{ssrA}$ snp noncoding',
        'ssuB T167M': 'SsuB T167M',
        'stfQ A193A': 'StfQ A193A',
        'sucA V141A': 'SucA V141A',
        'talB L236Q': 'TalB L236Q',
        'talB L38P': 'TalB L38P',
        'talB Q306*': 'TalB Q306*',
        'talB small_indel': '$\\it{talB}$ indel',
        'tamA A419V': 'TamA A419V',
        'tamA E47K': 'TamA E47K',
        'thiP G119D': 'ThiP G119D',
        'thpR A148V': 'ThpR A148V',
        'tktA mobile_element_insertion': '$\\it{tktA}$ IS insertion',
        'topA *866L': 'TopA *866L',
        'topA F780L': 'TopA F780L',
        'topA L123Q': 'TopA L123Q',
        'torC mobile_element_insertion': '$\\it{torC}$ IS insertion',
        'treC Q497R': 'TreC Q497R',
        'trkA I330T': 'TrkA I330T',
        'trkG G159D': 'TrkG G159D',
        'trkH G156C': 'TrkH G156C',
        'trkH L185Q': 'TrkH L185Q',
        'trkH L80Q': 'TrkH L80Q',
        'trkH Q159H': 'TrkH Q159H',
        'trkH S105Y': 'TrkH S105Y',
        'trkH T20I': 'TrkH T20I',
        'trkH T20P': 'TrkH T20P',
        'trkH V155E': 'TrkH V155E',
        'tufB D110Y': 'TufB D110Y',
        'uacT T314M': 'UacT T314M',
        'ugpE V127V': 'UgpE V127V',
        'uspE V298A': 'UspE V298A',
        'uvrC V593E': 'UvrC V593E',
        'waaF D13E': 'WaaF D13E',
        'xerC V56V': 'XerC V56V',
        'yaaA I52T': 'YaaA I52T',
        'yaaA S53I': 'Yaaa S53I',
        'yacH A495T': 'YacH A495T',
        'yagN mobile_element_insertion': '$\\it{yagN}$ IS insertion',
        'yahD N99N': 'YahD N99N',
        'yaiT–[yaiW] large_deletion': '$\\it{yaiT/yaiW}$ large deletion',
        'ybaL small_indel': '$\\it{ybaL}$ indel',
        'ybgS V45V': 'YbgS V45V',
        'yceM V88V': 'YceM V88V',
        'ycfH/ptsG mobile_element_insertion': '$\\it{ycfH/ptsG}$ ISinsertion',
        'ycgB S496N': 'YcgB S496N',
        'ycgM M111V': 'YcgM M111V',
        'ydcC mobile_element_insertion': '$\\it{ydcC}$ IS insertion',
        'ydcL A106T': 'YdcL A106T',
        'ydcL I105T': 'YdcL I105T',
        'ydeE I384V': 'YdeE I384V',
        'ydeE mobile_element_insertion': '$\\it{ydeE}$ IS insertion',
        'ydfD/ynfP snp_intergenic': '$\\it{ydfD/ynfP}$ snp intergenic',
        'ydfU C276R': 'YdfU C276R',
        'ydgI G84D': 'YdgI G84D',
        'ydgI S8S': 'YdgI S8S',
        'ydgJ R164C': 'YdgJ R164C',
        'ydhL A68V': 'YdhL A68V',
        'yeaD/yeaE snp_intergenic': '$\\it{yeaD/yeaE}$ snp intergenic',
        'yegT G58D': 'YegT G58D',
        'yfaQ K231E': 'YfaQ K231E',
        'yfbS A38S': 'YfbS A38S',
        'yfbS V35F': 'YfbS V35F',
        'yfcA L174L': 'YfcA L174L',
        'yfdS V98V': 'YfdS V98V',
        'yfeH A115V': 'YfeH A115V',
        'yfjH N8K': 'YfjH N8K',
        'ygcP/ygcQ snp_intergenic': '$\\it{ygcP/ygcQ}$ snp intergenic',
        'ygfB P184Q': 'YgfB P184Q',
        'ygfT G437G': 'YgfT G437G',
        'ygjI R466R': 'YgjI R466R',
        'ygjK A257T': 'YgjK A257T',
        'yhcF/yhcG mobile_element_insertion': '$\\it{yhcF/yhcG}$ IS insertion',
        'yhgH L211V': 'YhgH L211V',
        'yhgN M182V': 'YhgN M182V',
        'yhhI E52G': 'YhhI E52G',
        'yhiI V38V': 'YhiI V38V',
        'yiaN D80E': 'YiaN D80E',
        'yicI N297N': 'YicI N297N',
        'yieH V107A': 'YieH V107A',
        'yieK V227A': 'YieK V227A',
        'yihQ Y177H': 'YihQ Y177H',
        'yjbB L14Q': 'YjbB L14Q',
        'yjdM C23R': 'YjdM C23R',
        'ynaI G144D': 'YnaI G144D',
        'ynaI P217L': 'YnaI P217L',
        'ynaI/insH4 small_indel': '$\\it{ynaI/insH4}$ indel',
        'ynaI/insH4 snp_intergenic': '$\\it{ynaI/insH4}$ snp intergenic',
        'ynaM A20A': 'YnaM A20A',
        'yncG H36R': 'YncG H36R',
        'ynfS/ydfT snp_intergenic': '$\\it{ynfS/ydfT}$ snp intergenic',
        'yobH T7A': 'YobH T7A',
        'yohP/dusC snp_intergenic': '$\\it{yohP/dusC}$ snp intergenic',
        'ypeA V74A': 'YpeA V74A',
        'yqeF A246V': 'YqeF A246V',
        'yqiG mobile_element_insertion': '$\\it{yqiG}$ IS insertion',
        'yraK G321G': 'YraK G321G',
        'ytfJ A73E': 'YtfJ A73E',
        'ytjB S210S': 'YtjB S210S',
        'zapE K247E': 'ZapE K247E',
        'zapG V111A': 'ZapG V111A'}



    # Check required columns
  
    required_columns = ['Lineage', 'population', 'label', 'last_freq']
    if not all(col in combined_df.columns for col in required_columns):
        raise ValueError(f"Missing required columns in combined_df: {set(required_columns) - set(combined_df.columns)}")
    
    combined_df['label'] = combined_df['label'].apply(lambda x: alt_labels[x] if x in alt_labels else x)
   
    # Reassign specific labels for phoQ based on gene_pos
    combined_df.loc[
        (combined_df['gene_name'] == 'phoQ') & (combined_df['gene_pos'] == 'coding (1265-1268/1461 nt)'),
        'label'
    ] = 'phoQ IS2 insertion'

    combined_df.loc[
        (combined_df['gene_name'] == 'phoQ') & (combined_df['gene_pos'] == 'coding (136-140/1461 nt)'),
        'label'
    ] = 'phoQ IS5 insertion'



########FILTERING##############################################################################################################################
    # Calculate gene_name_count
    gene_name_counts = combined_df['gene_name'].value_counts().to_dict()
    combined_df['gene_name_count'] = combined_df['gene_name'].map(gene_name_counts)

    # Filter labels based on conditions
    combined_df = combined_df[~((combined_df['gene_name_count'] == 1) & (combined_df['last_freq'] < .3))]
    
    # Remove populations 2, 3, 8 from lineage PLAC
    populations_to_remove = ['PLAC_02', 'PLAC_03', 'PLAC_08']
    combined_df = combined_df[
        ~((combined_df['population'].isin(populations_to_remove)))
    ]

    # Calculate the maximum `last_freq` for each label
    max_last_freq_per_label = combined_df.groupby('label')['last_freq'].max()
    # Filter labels with max `last_freq` >= 0.2
    valid_labels = max_last_freq_per_label[max_last_freq_per_label >= 0.1].index
    # Filter the DataFrame to include only valid labels
    combined_df = combined_df[combined_df['label'].isin(valid_labels)]




########FILTERING##############################################################################################################################
    combined_df['Lineage_Pop'] = combined_df['population']

########SORTING##############################################################################################################################
    # Use population directly for x-axis labels
    # Define the custom order for the lineages
    custom_order = ['PL','PA', 'PLA','PLAC','PC']
    combined_df['Lineage_Rank'] = combined_df['Lineage'].apply(lambda x: custom_order.index(x) if x in custom_order else len(custom_order))


      # Custom manual sorting
    manual_gene_order = [
        'rpoB', 'gyrA','icd',  'rplJ', 'feaB', 'puuA', 
        'elaD', 'recF', 'paaG', 'pssL','arcA', 'arpA', 
        'tas', 'appY/ompT', 'insB9–[crl]', 'yhiI',  
        'yhaC/rnpB',  'sucA', 'ygfB', 'gltP/yjcO', 
        'selB', 'glvC', 'fimB/fimE', 
        'ftp/ompC','arcB', 'hlyE', 'yffN/yffO', 'trkH',
        
        'fusA', 'xerC', 'osmY', 'yaaA', 'yacH', 'glpF',
        'uvrC', 'pgsA', 'baeS', 'tufB', 'bluF', 
        
        'pitA','manZ/yobD', 'aceK','corA', 'mgtL/mgtA', 
        'yicC', 'eutH','relB','mrcB|mrcB','yfbS', 
        'phoQ','cstA', 'abgT', 'pgaC', 'prs',
        'glnH','yjbB', 'nadK','yhfG/ppiA','waaF',
        'cysS', 'marR', 'pheS', 'rbsA',
        
        'lon', 'glcB', 'yqeF', 'yieK', 'thiP', 'ptsI', 'fadB',
        'eutQ', 'yohP/dusC', 'mcrB/symE', 'adiY/adiA', 'torC', 
        'dnaQ', 'topA', 'rpoD', 'nikA', 'aroG', 'rssB/galU', 
        'iscR/trmJ', 'mutL', 'gadF/mdtE', 'rihB', 
        '[metR]–[ysgA]', 'serW', 'fimE/fimA', 'speE', 'clsB',  'cpxA', 
         'sbmA','ampH/sbmA', 'rnb',
        'mscM', 'narU/yddK', 'cyaA', 'thpR', 'ppa/ytfQ', 'yagN',  'hipA',
         'rpoZ', 'ftsH', 'gatA', 'mltB', 'ydcL', 'hns/tdk', 'ydgI', 'ptsG',
          'glpT', 'yebK', 'yfjH',  'tyrR',
           
           
    ]

    # Assign ranks based on manual_gene_order
    combined_df['gene_name_rank'] = combined_df['gene_name'].apply(
        lambda x: manual_gene_order.index(x) if x in manual_gene_order else len(manual_gene_order)
    )
    # Sort combined_df by gene_name_rank
    combined_df = combined_df.sort_values('gene_name_rank')

    # Align labels with `gene_name`
    if 'gene_name_rank' in combined_df.columns:
        label_order = (
            combined_df[['label', 'gene_name', 'gene_name_rank']]
            .drop_duplicates()
            .sort_values(by='gene_name_rank')['label']
            .tolist()
        )
    else:
        raise KeyError("The column 'gene_name_rank' is missing or was not created.")


    # Pivot for heatmap
    heatmap_data = combined_df.pivot_table(index='Lineage_Pop', columns='label', values='last_freq', fill_value=0)
    heatmap_data = heatmap_data[label_order] 
    heatmap_data = heatmap_data.loc[:, heatmap_data.sum(axis=0) > 0]  # Filter columns where sum of last_freq is 0
    sorted_index = combined_df.sort_values(['Lineage_Rank', 'population'])['Lineage_Pop'].unique()
    heatmap_data = heatmap_data.loc[sorted_index]
    # heatmap_data = heatmap_data.loc[
    #     heatmap_data.index,
    #     heatmap_data.sum(axis=0).sort_values(ascending=False).index
    # ]
   ########SORTING##############################################################################################################################

   


    # Replace the index with numeric portions and transpose the heatmap
    heatmap_data.index = heatmap_data.index.map(lambda x: re.search(r'\d+', x).group().lstrip('0') if re.search(r'\d+', x) else x)
    heatmap_data = heatmap_data.T  # Transpose to switch axes

    # Progress message
    print(f"Generating heatmap with {heatmap_data.shape[0]} mutations and {heatmap_data.shape[1]} populations.")

    # Create a custom colormap with white for 0 values
    standard_map = plt.cm.get_cmap('cool')
    new_colors = standard_map(np.linspace(0, 1, 256))
    new_colors[0] = np.array([1, 1, 1, 1])  # Replace the first color with white
    custom_map = mcolors.ListedColormap(new_colors)

    # Create the figure and gridspec for positioning the color scale
    fig = plt.figure(figsize=(10, 20))
    gs = gridspec.GridSpec(1, 2, width_ratios=[20, .5], wspace=0.01)

    ax = fig.add_subplot(gs[0])  # Heatmap axis
    cbar_ax = fig.add_subplot(gs[1])  # Color scale axis

    # Create the heatmap with the custom colormap
    sns.heatmap(
        heatmap_data,
        cmap=custom_map,  # Use the custom colormap
        annot=False,
        fmt=".2f",
        ax=ax,
        cbar_ax=cbar_ax,  # Place color bar in a separate axis
        cbar_kws={'shrink': 0.8, 'aspect': 20},  # Adjust color bar size
        yticklabels=True,
        xticklabels=True,
        vmin=0,
        vmax=1,    
        linewidths=0.5,
        linecolor='lightgrey'
    )
    # Add lineage names below the x-tick labels
    # num_ticks = len(heatmap_data.columns)  # Number of x-ticks
    # for i, lineage in enumerate(custom_order):
    #     position = (i + 0.5) / (num_ticks/9.5)  # Evenly space lineage names along the x-axis
    #     ax.text(
    #         x=position,  # Align with x-axis tick
    #         y=-0.015,  # Position below the x-axis
    #         s=lineage,  # Lineage name
    #         fontsize=12,
    #         ha='center',  # Center text horizontally
    #         va='top',  # Align text to the top of the position
    #         rotation=0,  # Keep text upright
    #         transform=ax.transAxes  # Use axis-relative coordinates
    #     )
    #     ax.text(
    #         x=position,  # Align with x-axis tick
    #         y=1.01,  # Position below the x-axis
    #         s=lineage,  # Lineage name
    #         fontsize=12,
    #         ha='center',  # Center text horizontally
    #         va='top',  # Align text to the top of the position
    #         rotation=0,  # Keep text upright
    #         transform=ax.transAxes  # Use axis-relative coordinates
    #     )
    # Set labels and title
    y_fontsize = max(6, 14 - heatmap_data.shape[0] // 10)
    x_fontsize = max(6, 14 - heatmap_data.shape[1] // 10)
    ax.set_ylabel('', fontsize=14)  # Y-axis now corresponds to mutations
    ax.set_xlabel('', fontsize=14)  # X-axis now corresponds to populations
    ax.set_title('', fontsize=16)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=y_fontsize)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0,  fontsize=x_fontsize)
    # Set plot spines
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(2)
        spine.set_color("black")
    plt.subplots_adjust(bottom=0.1)
    # Add vertical lines at specific x-values
    x_positions = [10, 20, 30,37]  # Specify the x-values where vertical lines are needed
    line_length = 160  # Specify the length of the lines in data coordinates

    for x in x_positions:
        ax.vlines(
            x=x,  # x-coordixnate for the line
            ymin=0,  # Starting y-coordinate
            ymax=line_length,  # Ending y-coordinate
            colors='black',  # Color of the line
            linestyles='solid',  # Line style (solid, dashed, etc.)
            linewidth=1  # Width of the line
        )

    # Save or show the plot
    plt.tight_layout()
    if save_plot:
        output_path = output_dir / file_name
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=600, bbox_inches='tight')
        print(f"Heatmap saved to {output_path}")
    else:
        plt.show()
    plt.show()
    # unique_gene_names = combined_df['gene_name'].unique()
    # print("Unique gene_name values:", unique_gene_names)
 
 

In [ ]:
lineages_of_interest = ['PL','PA','PLA','PLAC','PC']

all_allele_frequencies = []  # To store data for all lineages

for lineage in lineages_of_interest:
    pops, timepoints = get_lineage_info(meta, lineage)
    for pop in pops:
        allele_data = traceAlleleFreq(pop, meta, PROJECT_PATH, min_freq=0.1, verbose = False)
        if allele_data.empty:
            #print(f"No data for population {pop} in lineage {lineage}. Skipping.")
            continue
        allele_data['Lineage'] = lineage  # Add lineage information
        all_allele_frequencies.append(allele_data)

# Check for column consistency across all DataFrames
if not all(allele_data.columns.equals(all_allele_frequencies[0].columns) for allele_data in all_allele_frequencies):
    raise ValueError("Mismatch in DataFrame columns across populations.")

# Combine all allele frequency data into one DataFrame
combined_df = pd.concat(all_allele_frequencies)

# Add the 'last_freq' column
combined_df['last_freq'] = combined_df['freq'].apply(lambda x: x[-1])


In [ ]:
output_dir = PROJECT_PATH / "figures/final"
plot_combined_heatmap(combined_df, output_dir, save_plot=True)


In [ ]:
full_label_list = list(combined_df['label'].unique())
full_label_list


In [ ]:


# Filter out labels that do not contain '$\\'
no_latex_labels = sorted([label for label in full_label_list if '$\\' not in label])

no_latex_labels


In [ ]:
# Provided list of labels (trimmed here for practical execution — assume full list is present)


# Suffixes that should trigger italicization of gene names
italic_suffixes = [
    'insertion', 'snp_intergenic', 'mobile_element_insertion', 'small_indel'
]

# Construct alt_labels_two using rules
alt_labels_two = {}
for label in no_latex_labels:
    gene = label.split()[0]
    suffix = label.replace(gene, '').strip()
    if any(suffix.endswith(suffix_type) for suffix_type in italic_suffixes):
        alt_labels_two[label] = rf"$\it{{{gene}}}$ {suffix}"
    else:
        alt_labels_two[label] = f"{gene.capitalize()} {suffix}"

alt_labels_two

In [ ]:
# Process new query list into a clean list
exclude_genes = [
    "hipA", "hipB", "gatA", "glvC", "selB", "rpoZ", "ftsH", "fimE", "fimA",
    "lpxC", "kdtA", "gyrA", "rpoB", "icd", "fusA", "trkH", "phoQ", "phoP", "prs",
    "cpxA", "relA", "spoT", "lon", "sulA", "yqgE", "ompC", "acrA", "pgaC", "ygfB",
    "sbmA", "rpoH", "sucA", "priA", "dnaC", "dnaT", "cysK", "mdtK", "yhaM", "rmf",
    "oxyR", "rpoS", "dksA", "gsk", "nadC", "purF", "ahpF", "katG", "poxB", "acs",
    "lsrA", "ptsG", "aceK"
]


# Provided full list from user message (copied and reformatted)
full_gene_list = [
    'rpoB', 'gyrA', 'icd', 'rplJ', 'feaB', 'puuA', 'elaD', 'recF', 'paaG', 'pssL',
    'arcA', 'arpA', 'tas', 'appY/ompT', 'insB9–[crl]', 'yhiI', 'yhaC/rnpB', 'sucA',
    'ygfB', 'gltP/yjcO', 'selB', 'glvC', 'fimB/fimE', 'ftp/ompC', 'arcB', 'hlyE',
    'yffN/yffO', 'trkH', 'fusA', 'xerC', 'osmY', 'yaaA', 'yacH', 'glpF', 'uvrC',
    'pgsA', 'baeS', 'tufB', 'bluF', 'pitA', 'manZ/yobD', 'aceK', 'corA', 'mgtL/mgtA',
    'yicC', 'eutH', 'relB', 'mrcB|mrcB', 'yfbS', 'phoQ', 'cstA', 'abgT', 'pgaC', 'prs',
    'glnH', 'yjbB', 'nadK', 'yhfG/ppiA', 'waaF', 'cysS', 'marR', 'pheS', 'rbsA', 'lon',
    'glcB', 'yqeF', 'yieK', 'thiP', 'ptsI', 'fadB', 'eutQ', 'yohP/dusC', 'mcrB/symE',
    'adiY/adiA', 'torC', 'dnaQ', 'topA', 'rpoD', 'nikA', 'aroG', 'rssB/galU',
    'iscR/trmJ', 'mutL', 'gadF/mdtE', 'rihB', '[metR]–[ysgA]', 'serW', 'fimE/fimA',
    'speE', 'clsB', 'cpxA', 'sbmA', 'ampH/sbmA', 'rnb', 'mscM', 'narU/yddK', 'cyaA',
    'thpR', 'ppa/ytfQ', 'yagN', 'hipA', 'rpoZ', 'ftsH', 'gatA', 'mltB', 'ydcL',
    'hns/tdk', 'ydgI', 'ptsG', 'glpT', 'yebK', 'yfjH', 'tyrR'
]


# Create the filtered list
list_filtered = [gene for gene in full_gene_list if gene not in exclude_genes]

# Output result

list_filtered
